In [6]:

# CELL 1: Import libraries and define file paths

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    balanced_accuracy_score
)

RAIN_FILE = Path("../data/processed/kelani/kelani_daily_rainfall_combined.xlsx")
WL_FILE = Path("../data/processed/kelani/kelani_hourly_waterlevel_combined.xlsx")

print("Rainfall file exists:", RAIN_FILE.exists())
print("Water-level file exists:", WL_FILE.exists())


Rainfall file exists: True
Water-level file exists: True


In [7]:

# CELL 2: Load the original workbooks

rain_raw = pd.read_excel(RAIN_FILE, engine="openpyxl")
wl_raw = pd.read_excel(WL_FILE, engine="openpyxl")

rain_raw.columns = rain_raw.columns.astype(str).str.strip()
wl_raw.columns = wl_raw.columns.astype(str).str.strip()

print("Rainfall shape:", rain_raw.shape)
print("Water-level shape:", wl_raw.shape)

display(rain_raw.head(3))
display(wl_raw.head(3))


Rainfall shape: (792, 36)
Water-level shape: (37978, 5)


,Location,YEARMONTH,YEAR,MONTH,D1,D2,D3,D4,D5,D6,...,D23,D24,D25,D26,D27,D28,D29,D30,D31,TOTAL
0,NORWOOD,14/15/04,2015,JAN,0.0,0.0,0.0,0.0,0.0,1.7,...,0.0,0.0,0.0,0.0,0.0,11.7,23.4,4.6,0.5,41.9
1,NORWOOD,14/15/05,2015,FEB,0.0,0.0,10.4,0.0,25.9,23.9,...,5.0,7.2,0.0,12.5,20.0,31.7,NaN,NaN,NaN,182.2
2,NORWOOD,14/15/06,2015,MAR,3.7,0.0,0.0,20.3,2.5,0.0,...,0.0,0.0,0.0,25.4,1.3,1.8,4.6,7.0,5.5,85.7


,Location,Year,Month,Date,Avg. WL
0,Norwood,1996,1,1996-01-01,1098.211667
1,Norwood,1996,1,1996-01-02,1098.253333
2,Norwood,1996,1,1996-01-03,1098.202917


In [8]:

# CELL 3: Convert monthly rainfall columns into daily records

month_map = {
    "JAN": 1, "FEB": 2, "MAR": 3, "APR": 4,
    "MAY": 5, "JUN": 6, "JUL": 7, "AUG": 8,
    "SEP": 9, "OCT": 10, "NOV": 11, "DEC": 12
}

rain = rain_raw.copy()

rain["Location"] = (
    rain["Location"].astype(str).str.strip().str.title()
)
rain["Location"] = rain["Location"].replace({
    "Kitulgala": "Kithulgala"
})

rain["MONTH"] = (
    rain["MONTH"].astype(str).str.strip().str.upper()
)
rain["Month_Number"] = rain["MONTH"].map(month_map)
rain["YEAR"] = pd.to_numeric(rain["YEAR"], errors="coerce")

if rain["Month_Number"].isna().any():
    raise ValueError("Unknown month abbreviation found in rainfall data.")

day_columns = [f"D{i}" for i in range(1, 32)]

rain_daily = rain.melt(
    id_vars=["Location", "YEAR", "Month_Number"],
    value_vars=day_columns,
    var_name="Day",
    value_name="Rainfall"
)

rain_daily["Day"] = (
    rain_daily["Day"].str.replace("D", "", regex=False).astype(int)
)

rain_daily["Date"] = pd.to_datetime(
    {
        "year": rain_daily["YEAR"],
        "month": rain_daily["Month_Number"],
        "day": rain_daily["Day"]
    },
    errors="coerce"
)

rain_daily["Rainfall"] = pd.to_numeric(
    rain_daily["Rainfall"], errors="coerce"
)

# Remove invalid dates, such as 31 February
rain_daily = rain_daily.dropna(
    subset=["Date", "Rainfall"]
).copy()

rain_daily = rain_daily[
    ["Location", "Date", "Rainfall"]
].sort_values(["Location", "Date"])

if rain_daily.duplicated(["Location", "Date"]).any():
    raise ValueError("Duplicate station/date records in rainfall data.")

print("Daily rainfall records:", len(rain_daily))
print("Rainfall date range:", rain_daily["Date"].min(),
      "to", rain_daily["Date"].max())

display(rain_daily.head())


Daily rainfall records: 24103
Rainfall date range: 2015-01-01 00:00:00 to 2025-12-31 00:00:00


,Location,Date,Rainfall
396,Deraniyagala,2015-01-01,0.0
1188,Deraniyagala,2015-01-02,0.0
1980,Deraniyagala,2015-01-03,0.0
2772,Deraniyagala,2015-01-04,0.0
3564,Deraniyagala,2015-01-05,0.0


In [9]:

# CELL 3: Convert monthly rainfall data into daily records

month_map = {
    "JAN": 1, "FEB": 2, "MAR": 3, "APR": 4,
    "MAY": 5, "JUN": 6, "JUL": 7, "AUG": 8,
    "SEP": 9, "OCT": 10, "NOV": 11, "DEC": 12
}

rain = rain_raw.copy()

rain["Location"] = (
    rain["Location"].astype(str).str.strip().str.title()
)
rain["Location"] = rain["Location"].replace({
    "Kitulgala": "Kithulgala"
})

rain["MONTH"] = rain["MONTH"].astype(str).str.strip().str.upper()
rain["Month_Number"] = rain["MONTH"].map(month_map)
rain["YEAR"] = pd.to_numeric(rain["YEAR"], errors="coerce")

if rain["Month_Number"].isna().any():
    raise ValueError("Unknown month abbreviation in rainfall data.")

day_columns = [f"D{i}" for i in range(1, 32)]

rain_daily = rain.melt(
    id_vars=["Location", "YEAR", "Month_Number"],
    value_vars=day_columns,
    var_name="Day",
    value_name="Rainfall"
)

rain_daily["Day"] = (
    rain_daily["Day"].str.replace("D", "", regex=False).astype(int)
)

rain_daily["Date"] = pd.to_datetime(
    {
        "year": rain_daily["YEAR"],
        "month": rain_daily["Month_Number"],
        "day": rain_daily["Day"]
    },
    errors="coerce"
)

rain_daily["Rainfall"] = pd.to_numeric(
    rain_daily["Rainfall"], errors="coerce"
)

rain_daily = rain_daily.dropna(
    subset=["Date", "Rainfall"]
).copy()

rain_daily = rain_daily[["Location", "Date", "Rainfall"]]

rain_daily = rain_daily.sort_values(
    ["Location", "Date"]
).reset_index(drop=True)

if rain_daily.duplicated(["Location", "Date"]).any():
    raise ValueError("Duplicate station/date records in rainfall data.")

print("Rainfall daily records:", len(rain_daily))
display(rain_daily.head())


Rainfall daily records: 24103


,Location,Date,Rainfall
0,Deraniyagala,2015-01-01,0.0
1,Deraniyagala,2015-01-02,0.0
2,Deraniyagala,2015-01-03,0.0
3,Deraniyagala,2015-01-04,0.0
4,Deraniyagala,2015-01-05,0.0


In [10]:

# CELL 5: Merge rainfall and water level by station and date

df = pd.merge(
    wl,
    rain_daily,
    on=["Location", "Date"],
    how="inner",
    validate="one_to_one"
)

df = df.sort_values(["Location", "Date"]).reset_index(drop=True)

print("Merged records:", len(df))
print("Merged date range:", df["Date"].min(), "to", df["Date"].max())

print("\nRecords per station:")
print(df.groupby("Location").size())

print("\nRecords per year:")
print(df.groupby(df["Date"].dt.year).size())


Merged records: 24103
Merged date range: 2015-01-01 00:00:00 to 2025-12-31 00:00:00

Records per station:
Location
Deraniyagala    4017
Glencourse      4018
Hanwella        4018
Holombuwa       4018
Kithulgala      4018
Norwood         4014
dtype: int64

Records per year:
Date
2015    2190
2016    2195
2017    2190
2018    2188
2019    2190
2020    2196
2021    2190
2022    2190
2023    2190
2024    2196
2025    2188
dtype: int64


In [11]:

# CELL 6: Filter stations and define official thresholds

thresholds = {
    "Hanwella": {
        "Alert": 6.50,
        "Minor": 7.50,
        "Major": 9.50
    },
    "Glencourse": {
        "Alert": 15.00,
        "Minor": 15.50,
        "Major": 19.00
    }
}

df = df[df["Location"].isin(thresholds)].copy()

print("Stations included:", sorted(df["Location"].unique()))
print("Records retained:", len(df))

# Assign tomorrow's water level to today's observation
df = df.sort_values(["Location", "Date"]).reset_index(drop=True)

grouped = df.groupby("Location", sort=False)

df["Next_Date"] = grouped["Date"].shift(-1)
df["Next_Water_Level"] = grouped["Water_Level"].shift(-1)

# Retain only consecutive days: today predicts tomorrow
df = df[
    df["Next_Date"] == df["Date"] + pd.Timedelta(days=1)
].copy()

# Features use today's and past observations only
grouped = df.groupby("Location", sort=False)

df["Rainfall_3day"] = grouped["Rainfall"].transform(
    lambda s: s.rolling(3, min_periods=3).sum()
)

df["Rainfall_7day"] = grouped["Rainfall"].transform(
    lambda s: s.rolling(7, min_periods=7).sum()
)

df["Water_Level_Change"] = grouped["Water_Level"].diff()

df = df.dropna(
    subset=[
        "Rainfall_3day",
        "Rainfall_7day",
        "Water_Level_Change",
        "Next_Water_Level"
    ]
).copy()

# Attach tomorrow's station-specific thresholds
df["Alert_Threshold"] = df["Location"].map(
    {s: v["Alert"] for s, v in thresholds.items()}
)
df["Minor_Threshold"] = df["Location"].map(
    {s: v["Minor"] for s, v in thresholds.items()}
)
df["Major_Threshold"] = df["Location"].map(
    {s: v["Major"] for s, v in thresholds.items()}
)

# The target is tomorrow's threshold category
df["Target_Severity"] = np.select(
    [
        df["Next_Water_Level"] >= df["Major_Threshold"],
        df["Next_Water_Level"] >= df["Minor_Threshold"],
        df["Next_Water_Level"] >= df["Alert_Threshold"]
    ],
    ["Major", "Minor", "Alert"],
    default="Low"
)

print("Forecasting records:", len(df))
print("\nTarget class counts:")
print(df["Target_Severity"].value_counts())

display(df[[
    "Date", "Next_Date", "Location",
    "Water_Level", "Rainfall",
    "Next_Water_Level", "Target_Severity"
]].head())


Stations included: ['Glencourse', 'Hanwella']
Records retained: 8036
Forecasting records: 8022

Target class counts:
Target_Severity
Low      7938
Minor      39
Alert      34
Major      11
Name: count, dtype: int64


,Date,Next_Date,Location,Water_Level,Rainfall,Next_Water_Level,Target_Severity
6,2015-01-07,2015-01-08,Glencourse,9.222290,0.0,9.125775,Low
7,2015-01-08,2015-01-09,Glencourse,9.125775,0.0,9.106726,Low
8,2015-01-09,2015-01-10,Glencourse,9.106726,0.0,9.055293,Low
9,2015-01-10,2015-01-11,Glencourse,9.055293,0.0,9.043864,Low
10,2015-01-11,2015-01-12,Glencourse,9.043864,0.0,9.055293,Low


In [12]:

# CELL 7: Chronological training and testing split

train = df[df["Next_Date"].dt.year.between(2015, 2024)].copy()
test = df[df["Next_Date"].dt.year == 2025].copy()

features = [
    "Location",
    "Water_Level",
    "Rainfall",
    "Rainfall_3day",
    "Rainfall_7day",
    "Water_Level_Change"
]

X_train = train[features]
y_train = train["Target_Severity"]

X_test = test[features]
y_test = test["Target_Severity"]

print("Training target years: 2015–2024")
print("Testing target year: 2025")
print("Training records:", len(train))
print("Testing records:", len(test))

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\n2025 target distribution:")
print(y_test.value_counts())

if train.empty or test.empty:
    raise ValueError("Training or testing data is empty.")


Training target years: 2015–2024
Testing target year: 2025
Training records: 7292
Testing records: 730

Training target distribution:
Target_Severity
Low      7216
Minor      37
Alert      34
Major       5
Name: count, dtype: int64

2025 target distribution:
Target_Severity
Low      722
Major      6
Minor      2
Name: count, dtype: int64


In [13]:

# CELL 8: Train Random Forest

preprocessor = ColumnTransformer(
    transformers=[
        (
            "station",
            OneHotEncoder(handle_unknown="ignore"),
            ["Location"]
        )
    ],
    remainder="passthrough"
)

flood_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        min_samples_leaf=3,
        n_jobs=-1
    ))
])

flood_model.fit(X_train, y_train)

predictions = flood_model.predict(X_test)

print("===== 2025 TEST RESULTS =====")
print(f"Accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%")
print(
    f"Balanced accuracy: "
    f"{balanced_accuracy_score(y_test, predictions) * 100:.2f}%"
)

print("\nClassification report:")
print(classification_report(
    y_test,
    predictions,
    labels=["Low", "Alert", "Minor", "Major"],
    zero_division=0
))

print("\nConfusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(
    y_test,
    predictions,
    labels=["Low", "Alert", "Minor", "Major"]
))

# Simple baseline: always predict the most common training class
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)

print(
    f"\nMajority baseline accuracy: "
    f"{accuracy_score(y_test, baseline_predictions) * 100:.2f}%"
)


===== 2025 TEST RESULTS =====
Accuracy: 98.77%
Balanced accuracy: 60.93%

Classification report:
              precision    recall  f1-score   support

         Low       1.00      0.99      1.00       722
       Alert       0.00      0.00      0.00         0
       Minor       0.25      0.50      0.33         2
       Major       1.00      0.33      0.50         6

    accuracy                           0.99       730
   macro avg       0.56      0.46      0.46       730
weighted avg       1.00      0.99      0.99       730


Confusion matrix (rows = actual, columns = predicted):
[[718   3   1   0]
 [  0   0   0   0]
 [  1   0   1   0]
 [  1   1   2   2]]

Majority baseline accuracy: 98.90%


/Users/amashasinhalagoda/Desktop/IT4010 - Research Project/flood-severity-assessment/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [14]:

# CELL 9: Save the trained model and full 2025 test results

import joblib

model_dir = Path("../models")
output_dir = Path("../data/processed/kelani")

model_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    flood_model,
    model_dir / "kelani_next_day_threshold_rf.joblib"
)

results_2025 = test[[
    "Date",
    "Next_Date",
    "Location",
    "Water_Level",
    "Rainfall",
    "Rainfall_3day",
    "Rainfall_7day",
    "Water_Level_Change",
    "Next_Water_Level",
    "Target_Severity"
]].copy()

results_2025["Predicted_Severity"] = predictions
results_2025["Correct"] = (
    results_2025["Target_Severity"]
    == results_2025["Predicted_Severity"]
)

results_2025.to_csv(
    output_dir / "kelani_next_day_threshold_predictions_2025.csv",
    index=False
)

print("Model saved.")
print("Full 2025 prediction results saved.")
display(results_2025.head(10))


Model saved.
Full 2025 prediction results saved.


,Date,Next_Date,Location,Water_Level,Rainfall,Rainfall_3day,Rainfall_7day,Water_Level_Change,Next_Water_Level,Target_Severity,Predicted_Severity,Correct
3652,2024-12-31,2025-01-01,Glencourse,8.629917,4.2,5.4,5.4,0.015000,8.542000,Low,Low,True
3653,2025-01-01,2025-01-02,Glencourse,8.542000,16.9,21.6,22.3,-0.087917,8.492833,Low,Low,True
3654,2025-01-02,2025-01-03,Glencourse,8.492833,0.5,21.6,22.8,-0.049167,8.485750,Low,Low,True
3655,2025-01-03,2025-01-04,Glencourse,8.485750,0.0,17.4,22.8,-0.007083,8.525333,Low,Low,True
3656,2025-01-04,2025-01-05,Glencourse,8.525333,0.0,0.5,22.8,0.039583,8.436167,Low,Low,True
3657,2025-01-05,2025-01-06,Glencourse,8.436167,0.0,0.0,22.1,-0.089167,8.501167,Low,Low,True
3658,2025-01-06,2025-01-07,Glencourse,8.501167,0.0,0.0,21.6,0.065000,8.494500,Low,Low,True
3659,2025-01-07,2025-01-08,Glencourse,8.494500,0.0,0.0,17.4,-0.006667,8.424500,Low,Low,True
3660,2025-01-08,2025-01-09,Glencourse,8.424500,0.0,0.0,0.5,-0.070000,8.404500,Low,Low,True
3661,2025-01-09,2025-01-10,Glencourse,8.404500,0.0,0.0,0.0,-0.020000,8.434917,Low,Low,True
